In [74]:
%pip install meteostat
%pip install scipy
%pip intall matplotlib.pyplot

Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 74.4 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.2
    Uninstalling numpy-2.4.2:
      Successfully uninstalled numpy-2.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
ERROR: unknown command "intall" - maybe you meant "install"
Note: you may need to restart the kernel to u

In [75]:
from datetime import date
import meteostat as ms
from meteostat import Point, daily
import pandas as pd
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple


In [76]:
# =============================================================================
# CONFIG
# =============================================================================

# 1) Station → departement mapping
STATION_TO_REG_PATH =  "../EDA/station_to_reg.csv"
# Expected columns: ["station_id", "departement_code"]

# 2) Date range for Meteostat request (change as needed)
START_DATE = date(2000, 1, 1)
END_DATE   = date(2024, 12, 31)

# 3) If you have precomputed weights per station & departement, plug them in here.
#    Otherwise, equal weights per (date, departement) are used.
USE_WEIGHTS = False
STATION_WEIGHTS_PATH = "../EDA/station_weights_by_region.csv"
# Expected columns: ["station_id", "departement_code", "weight"]

In [77]:

# =============================================================================
# DATA STRUCTURES
# =============================================================================

@dataclass
class OUEstimate:
    region: str
    kappa: float      # speed of mean reversion
    mu: float         # long-run mean (constant for this simple version)
    sigma: float      # diffusion (residual std)




In [78]:
# =============================================================================
# HELPERS: LOADING & FILTERS
# =============================================================================

def load_station_to_region(path: Path) -> pd.DataFrame:
    """
    Load station→region mapping and filter to mainland France
    by excluding region codes starting with 97 or 98.

    CSV columns expected: station_id, region_code
    """
    df = pd.read_csv(path, dtype={"station_id": str, "region_code": str})

    # Filter to mainland: exclude overseas regions (97*, 98*)
    mask_overseas = df["region_code"].str.startswith(("97", "98"))
    df_mainland = df[~mask_overseas].copy()

    # Nothing to rename: already station_id, region_code
    return df_mainland


def load_station_weights(path: Path) -> pd.DataFrame:
    """
    Optional: load station weights per region (departement),
    normalized to 1 per region.

    CSV columns expected: region_code, station_id, distance_m, weight
    """
    df = pd.read_csv(path, dtype={"station_id": str, "region_code": str})

    # Keep only what we actually need for aggregation
    if "weight" not in df.columns:
        raise ValueError("weights CSV must have a 'weight' column")

    df["weight"] = df["weight"].astype(float)
    df = df[df["weight"] > 0]

    # Normalize weights so they sum to 1 per region
    df["weight"] = df.groupby("region_code")["weight"].transform(lambda x: x / x.sum())

    # Return only the columns used later in merges
    return df[["station_id", "region_code", "weight"]]



In [87]:
# =============================================================================
# METEOSTAT → STATION DAILY TEMP
# =============================================================================

def fetch_station_daily_temps(
    station_ids: list,
    start,
    end,
) -> pd.DataFrame:
    all_frames = []

    for sid in station_ids:
        print(f"Fetching Meteostat daily data for station {sid}...")
        df = daily(sid, start, end)   # NOTE: no .fetch()

        if df is None or df.empty:
            print(f"  Warning: no data for station {sid} in given period.")
            continue

        if "tavg" not in df.columns:
            print(f"  Warning: no 'tavg' column for station {sid}, skipping.")
            continue

        out = (
            df[["tavg"]]
            .reset_index()
            .rename(columns={"time": "date", "tavg": "tmean"})
        )
        out["station_id"] = sid
        all_frames.append(out)

    if not all_frames:
        raise ValueError("No station data fetched from Meteostat. Check IDs & date range.")

    temp_df = pd.concat(all_frames, ignore_index=True)
    temp_df["date"] = pd.to_datetime(temp_df["date"])
    temp_df["station_id"] = temp_df["station_id"].astype(str)

    return temp_df[["date", "station_id", "tmean"]].dropna()
            .reset_index()  # index is 'time'
            .rename(columns={"time": "date", "tavg": "tmean"})
        )
        out["station_id"] = sid
        all_frames.append(out)

    if not all_frames:
        raise ValueError("No station data fetched from Meteostat. Check IDs & date range.")

    temp_df = pd.concat(all_frames, ignore_index=True)
    temp_df["date"] = pd.to_datetime(temp_df["date"])
    temp_df["station_id"] = temp_df["station_id"].astype(str)

    return temp_df[["date", "station_id", "tmean"]].dropna()

IndentationError: unindent does not match any outer indentation level (<string>, line 42)

In [80]:

# =============================================================================
# REGION AGGREGATION
# =============================================================================

def build_region_daily_temperature(
    temp_df: pd.DataFrame,
    station_to_reg: pd.DataFrame,
    weights_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """
    Build daily region-level temperatures using station→region mapping and optional weights.

    - Only mainland regions from station_to_reg are used.
    - Equal weights per (date, region) if weights_df is None.

    Returns DataFrame: [date, region_code, T_region]
    """
    df = temp_df.copy()

    # Map station → region (departement)
    df = df.merge(station_to_reg, on="station_id", how="inner")
    # At this point, station_to_reg is already filtered to mainland

    # Apply weights
    if weights_df is not None:
        df = df.merge(
            weights_df[["station_id", "region_code", "weight"]],
            on=["station_id", "region_code"],
            how="left",
        )
        if df["weight"].isna().any():
            raise ValueError("Some station-region pairs have no weight assigned.")
    else:
        # Equal weights for stations within each (date, region)
        df["weight"] = 1.0
        df["weight"] = df["weight"] / df.groupby(
            ["date", "region_code"]
        )["weight"].transform("sum")

    df["weighted_temp"] = df["weight"] * df["tmean"]

    region_daily = (
        df.groupby(["date", "region_code"], as_index=False)["weighted_temp"]
          .sum()
          .rename(columns={"weighted_temp": "T_region"})
    )

    return region_daily


In [81]:

# =============================================================================
# SIMPLE OU (AR(1) DISCRETE) ESTIMATION
# =============================================================================

def estimate_ou_from_series(series: pd.Series, dt: float = 1.0) -> OUEstimate:
    """
    Estimate OU parameters from a daily temperature series using AR(1) on levels:

        T_{t+dt} - mu = phi * (T_t - mu) + eps
        phi = exp(-kappa * dt)

    This is a simplified constant-mean, constant-variance OU.
    """
    x = series.dropna().values
    if len(x) < 3:
        raise ValueError("Series too short for OU estimation.")

    x_t = x[:-1]
    x_tp1 = x[1:]

    mu_hat = x.mean()

    y_t = x_t - mu_hat
    y_tp1 = x_tp1 - mu_hat

    phi_hat = np.sum(y_t * y_tp1) / np.sum(y_t ** 2)

    if phi_hat <= 0:
        kappa_hat = 1e-3
    else:
        kappa_hat = -np.log(phi_hat) / dt

    eps_hat = y_tp1 - phi_hat * y_t
    sigma_hat = np.std(eps_hat, ddof=1) / np.sqrt(dt)

    return OUEstimate(region="", kappa=kappa_hat, mu=mu_hat, sigma=sigma_hat)


def estimate_ou_by_region(region_daily: pd.DataFrame) -> Dict[str, OUEstimate]:
    """
    Estimate OU parameters for each region_code in region_daily.

    region_daily: [date, region_code, T_region]
    """
    ests: Dict[str, OUEstimate] = {}
    for reg_code, sub in region_daily.groupby("region_code"):
        sub = sub.sort_values("date")
        ser = sub["T_region"]
        est = estimate_ou_from_series(ser)
        est.region = reg_code
        ests[reg_code] = est
    return ests


In [82]:


# =============================================================================
# HDD / CDD / CAT INDEX HELPERS
# =============================================================================

def compute_hdd_series(T: pd.Series, base_temp: float = 18.0) -> pd.Series:
    return np.maximum(base_temp - T, 0.0)


def compute_cdd_series(T: pd.Series, base_temp: float = 18.0) -> pd.Series:
    return np.maximum(T - base_temp, 0.0)


def compute_cat_series(T: pd.Series) -> pd.Series:
    return T.cumsum()


In [83]:


# =============================================================================
# MAIN PIPELINE
# =============================================================================

def main():
    # 1) Load station→region mapping (and filter to mainland)
    print("Loading station→departement mapping and filtering to mainland France...")
    station_to_reg = load_station_to_region(STATION_TO_REG_PATH)
    print(f"Mainland mapping has {station_to_reg['station_id'].nunique()} stations "
          f"across {station_to_reg['region_code'].nunique()} departements.")

    # 2) Fetch Meteostat data for those stations
    station_ids = sorted(station_to_reg["station_id"].unique().tolist())
    temp_df = fetch_station_daily_temps(
        station_ids=station_ids,
        start=START_DATE,
        end=END_DATE,
    )
    out_station_daily = "../stations_daily_temp_mainland.csv"
    temp_df.to_csv(out_station_daily, index=False)
    print(f"Saved raw station daily temps to {out_station_daily}")

    # 3) Load weights if available
    if USE_WEIGHTS:
        print("Loading station weights...")
        weights_df = load_station_weights(STATION_WEIGHTS_PATH)
    else:
        print("Not using weights — equal weights per R.")
        weights_df = None

    # 4) Build region-level daily temperature
    print("Aggregating to departement-level daily temperatures...")
    region_daily = build_region_daily_temperature(
        temp_df=temp_df,
        station_to_reg=station_to_reg,
        weights_df=weights_df,
    )
    out_region = "../region_daily_temperature_mainland.csv"
    region_daily.to_csv(out_region, index=False)
    print(f"Saved region series to {out_region}")

    # 5) Estimate OU parameters per region
    print("Estimating OU parameters per departement (AR(1) on levels)...")
    ou_estimates = estimate_ou_by_region(region_daily)
    for reg, est in sorted(ou_estimates.items(), key=lambda x: x[0]):
        print(f"Departement {reg}: kappa={est.kappa:.4f}, mu={est.mu:.2f}, sigma={est.sigma:.2f}")

    # 6) Example index computation for one region (for sanity check)
    example_region = next(iter(ou_estimates.keys()))
    print(f"\nExample HDD/CAT computation for departement {example_region}...")
    sub = region_daily[region_daily["region_code"] == example_region].sort_values("date")
    T_reg = sub["T_region"]
    sub = sub.copy()
    sub["HDD"] = compute_hdd_series(T_reg)
    sub["CAT_cumsum"] = compute_cat_series(T_reg)
    sample_out = f"region_{example_region}_with_indices_sample.csv"
    sub.to_csv(sample_out, index=False)
    print(f"Saved sample indices for departement {example_region} to {sample_out}")


if __name__ == "__main__":
    main()



Could not fetch data for provider "daily"
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/pandas/io/pickle.py", line 202, in read_pickle
    return pickle.load(handles.handle)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/numpy/_core/numeric.py", line 11, in <module>
    from . import multiarray
  File "/opt/anaconda3/lib/python3.12/site-packages/numpy/_core/multiarray.py", line 97, in <module>
    _override___module__()
  File "/opt/anaconda3/lib/python3.12/site-packages/numpy/_core/multiarray.py", line 93, in _override___module__
    ufunc.__module__ = "numpy"
    ^^^^^^^^^^^^^^^^
AttributeError: 'numpy.ufunc' object has no attribute '__module__'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/meteostat/core/data.py", line 98, in _fetch_provider_data
    df = provider_service.fetch_data(provide

Loading station→departement mapping and filtering to mainland France...
Mainland mapping has 197 stations across 13 departements.
Fetching Meteostat daily data for station 07002...


AttributeError: 'NoneType' object has no attribute 'empty'